In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Milestone - 4**

# **Setup**

In [2]:
!pip install transformers peft datasets -q

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

print("Setup done.")
print(train.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setup done.
(2000, 8)


Q 1) **Label encoding for row index 150**

In [3]:
train['label'] = train['answer'].map(LABEL_MAP)

label_150 = train.iloc[150]['label']
print(f"Answer Q1 — Encoded label for row 150: {int(label_150)}")

Answer Q1 — Encoded label for row 150: 2


Q 2) **Character length of Option B formatted input for row 0**  

In [4]:
row0 = train.iloc[0]

formatted_B = str(row0['prompt']) + " [SEP] " + str(row0['B'])

print(f"Formatted string: {formatted_B}")
print(f"Answer Q2 — Character length: {len(formatted_B)}")

Formatted string: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Answer Q2 — Character length: 407


Q 3) **Second dimension of [1, 5, 128] tensor**

In [5]:
row0 = train.iloc[0]

formatted_inputs = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTION_COLS]

encoded = tokenizer(
    formatted_inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids = encoded['input_ids'].unsqueeze(0)  # [1, 5, 128]
print(f"Shape: {input_ids.shape}")
print(f"Answer Q3 — Second dimension: {input_ids.shape[1]}")

Shape: torch.Size([1, 5, 128])
Answer Q3 — Second dimension: 5


Q 4) **Total token positions in [16, 5, 128] tensor**

In [6]:
all_input_ids = []

for i in range(16):
    row = train.iloc[i]
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in OPTION_COLS]
    enc = tokenizer(
        formatted,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    all_input_ids.append(enc['input_ids'])

batch_input_ids = torch.stack(all_input_ids)  # [16, 5, 128]
print(f"Shape: {batch_input_ids.shape}")

total_tokens = batch_input_ids.numel()
print(f"Answer Q4 — Total token positions: {total_tokens}")

Shape: torch.Size([16, 5, 128])
Answer Q4 — Total token positions: 10240


Q 5) **Number of logits for one question**

In [7]:
model_mc = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
model_mc.eval()

row0 = train.iloc[0]
formatted = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTION_COLS]
enc = tokenizer(
    formatted,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids      = enc['input_ids'].unsqueeze(0)
attention_mask = enc['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask)

print(f"Logits shape: {outputs.logits.shape}")
print(f"Answer Q5 — Number of logits: {outputs.logits.shape[1]}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Answer Q5 — Number of logits: 5


Q 6) **Number of dimensions of loss tensor**

In [8]:
label_0 = torch.tensor([LABEL_MAP[train.iloc[0]['answer']]])

with torch.no_grad():
    outputs_with_loss = model_mc(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=label_0
    )

loss = outputs_with_loss.loss
print(f"Loss value : {loss.item()}")
print(f"Loss shape : {loss.shape}")
print(f"Answer Q6 — Number of dimensions: {loss.dim()}")

Loss value : 1.6073803901672363
Loss shape : torch.Size([])
Answer Q6 — Number of dimensions: 0


Q 7) **LoRA trainable parameters**

In [9]:
model_lora = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model_lora = get_peft_model(model_lora, lora_config)

trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Answer Q7 — Trainable parameters: {trainable_params}")
model_lora.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Answer Q7 — Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


Q 8) **Number of tokenized choices in first dataset item**

In [13]:
def tokenize_row(row):
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in OPTION_COLS]
    enc = tokenizer(
        formatted,
        padding='max_length',
        truncation=True,
        max_length=128
    )
    return {
        'input_ids'      : enc['input_ids'],
        'attention_mask' : enc['attention_mask'],
        'labels'         : LABEL_MAP[row['answer']]
    }

first_100 = train.iloc[:100]
data_list = [tokenize_row(row) for _, row in first_100.iterrows()]

hf_dataset = Dataset.from_list(data_list)

first_item = hf_dataset[0]
input_ids_shape = np.array(first_item['input_ids']).shape
print(f"input_ids shape: {input_ids_shape}")
print(f"Answer Q8 — Number of tokenized choices: {input_ids_shape[0]}")

input_ids shape: (5, 128)
Answer Q8 — Number of tokenized choices: 5


Q9)  **Final global_step after Tiny LoRA fine-tuning**

In [14]:
first_32  = train.iloc[:32]
data_32   = [tokenize_row_64(row) for _, row in first_32.iterrows()]
hf_32     = Dataset.from_list(data_32)
hf_32.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

model_ft = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_config_ft = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
model_ft = get_peft_model(model_ft, lora_config_ft)

training_args = TrainingArguments(
    output_dir                  = './results',
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps                   = 4,
    logging_steps               = 1,
    save_steps                  = 100,
    use_cpu                     = True
)

trainer = Trainer(
    model         = model_ft,
    args          = training_args,
    train_dataset = hf_32
)

trainer.train()
print(f"Answer Q9 — Final global_step: {trainer.state.global_step}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,1.643152
2,1.586063
3,1.601940
4,1.624213


Answer Q9 — Final global_step: 4


Q10) **Probability of Option E after fine-tuning**

In [15]:
model_ft.eval()

row0 = train.iloc[0]
formatted_0 = [str(row0['prompt']) + " [SEP] " + str(row0[opt]) for opt in OPTION_COLS]
enc_0 = tokenizer(
    formatted_0,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt'
)

input_ids_0      = enc_0['input_ids'].unsqueeze(0)
attention_mask_0 = enc_0['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs_ft = model_ft(input_ids=input_ids_0, attention_mask=attention_mask_0)

logits = outputs_ft.logits  # [1, 5]
probs  = torch.softmax(logits, dim=-1)[0]

print("Probabilities:")
for opt, prob in zip(OPTION_COLS, probs):
    print(f"  Option {opt}: {prob.item():.4f}")

print(f"\nAnswer Q10 — Probability of Option E: {probs[4].item():.4f}")

Probabilities:
  Option A: 0.2047
  Option B: 0.1953
  Option C: 0.1946
  Option D: 0.2031
  Option E: 0.2023

Answer Q10 — Probability of Option E: 0.2023
